# 0liviA — Kaggle T4×2 offline benchmark

**Uso exclusivo:** benchmark privado, manual y acotado de modelos ML con datos sintéticos.

**No es un endpoint:** esta notebook no inicia un servidor, túnel, proxy, heartbeat ni conexión con 0liviA / GitHub Actions. No cargar chats, secretos, credenciales ni datos privados.

Antes de activar GPU: revisar AUP/Terms y cuota real de la cuenta. Hardware y cuota pueden no estar disponibles en cada sesión.

In [ ]:
import json, os, platform, subprocess, sys, time, hashlib, tempfile
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Notebook purpose: offline ML benchmark only")
print("No HTTP server / tunnel / Core connection is part of this notebook.")
mem_kib = next((int(line.split()[1]) for line in Path("/proc/meminfo").read_text().splitlines()
                if line.startswith("MemTotal:")), None)
print({"cpu_count": os.cpu_count(), "memory_total_gib": round(mem_kib / 1024**2, 2) if mem_kib else None})

In [ ]:
LLAMA_CPP_SHA = "2cbca389f515df455412dafa07e2b1359cc72a6d"
CUDA_ARCH = "75"
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, *, check=True, capture=False, timeout=None):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=check,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
        timeout=timeout,
    )
    if capture and result.stderr:
        print("stderr:", result.stderr[-2000:])
    return result

def run_monitored(cmd, *, timeout=1800, interval=2):
    """Run one offline inference process and sample both GPUs until it exits."""
    print("$", " ".join(map(str, cmd)))
    samples = []
    started = time.monotonic()
    with tempfile.TemporaryFile(mode="w+t", encoding="utf-8") as stdout_file, \
         tempfile.TemporaryFile(mode="w+t", encoding="utf-8") as stderr_file:
        proc = subprocess.Popen(
            list(map(str, cmd)), text=True, stdout=stdout_file, stderr=stderr_file
        )
        try:
            while proc.poll() is None:
                elapsed = time.monotonic() - started
                if elapsed > timeout:
                    proc.terminate()
                    try:
                        proc.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        proc.kill()
                        proc.wait()
                    raise TimeoutError(f"Process exceeded {timeout}s: {cmd[0]}")
                snap = subprocess.run(
                    ["nvidia-smi", "--query-gpu=memory.used,memory.free",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, check=False, timeout=5
                )
                if snap.returncode == 0:
                    gpus = []
                    for line in snap.stdout.splitlines():
                        fields = [part.strip() for part in line.split(",")]
                        if len(fields) == 2:
                            try:
                                gpus.append({"used_mib": int(fields[0]), "free_mib": int(fields[1])})
                            except ValueError:
                                pass
                    if len(gpus) == 2:
                        samples.append({"elapsed_seconds": round(elapsed, 2), "gpus": gpus})
                time.sleep(interval)
            stdout_file.seek(0)
            stderr_file.seek(0)
            return type("RunResult", (), {
                "returncode": proc.returncode,
                "stdout": stdout_file.read(),
                "stderr": stderr_file.read(),
                "gpu_samples": samples,
                "wall_seconds": round(time.monotonic() - started, 3),
            })()
        finally:
            if proc.poll() is None:
                proc.kill()
                proc.wait()

gpu = run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"], capture=True)
print(gpu.stdout)
import torch
assert torch.cuda.is_available(), "CUDA is unavailable; stop before building/loading."
assert torch.cuda.device_count() == 2, f"Expected 2 GPUs, got {torch.cuda.device_count()}"
names = [torch.cuda.get_device_name(i) for i in range(2)]
caps = [torch.cuda.get_device_capability(i) for i in range(2)]
assert all("T4" in n for n in names), f"Expected Tesla T4 x2, got {names}"
assert all(c == (7, 5) for c in caps), f"Expected SM 7.5, got {caps}"
print({"gpu_names": names, "compute_capability": caps, "cpu_count": os.cpu_count()})

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
manifests = list(INPUT_ROOT.rglob("manifest.json"))
assert len(manifests) == 1, (
    f"Expected exactly one attached model manifest under {INPUT_ROOT}; found {len(manifests)}. "
    "Attach one private dataset containing one GGUF and one manifest.json."
)
MANIFEST_PATH = manifests[0]
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
required = ["model_id", "source_url", "license", "quantization", "file", "sha256"]
missing = [k for k in required if not manifest.get(k)]
assert not missing, f"Manifest missing required fields: {missing}"
MODEL_PATH = MANIFEST_PATH.parent / manifest["file"]
assert MODEL_PATH.is_file(), f"GGUF not found: {MODEL_PATH}"
print("Model:", manifest["model_id"])
print("Source:", manifest["source_url"])
print("License:", manifest["license"])
print("Quant:", manifest["quantization"])
print("File size GiB:", round(MODEL_PATH.stat().st_size / 1024**3, 2))

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

actual_sha = sha256_file(MODEL_PATH)
expected_sha = str(manifest["sha256"]).lower().strip()
assert len(expected_sha) == 64 and all(c in "0123456789abcdef" for c in expected_sha), "Manifest SHA-256 is not a real 64-hex digest."
assert actual_sha == expected_sha, f"SHA-256 mismatch: expected {expected_sha}, got {actual_sha}"
print("SHA256 verified:", actual_sha)

In [ ]:
LLAMA_DIR = Path("/kaggle/working/llama.cpp")
if not LLAMA_DIR.exists():
    run(["git", "clone", "--filter=blob:none", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_DIR)])
run(["git", "-C", LLAMA_DIR, "fetch", "--depth=1", "origin", LLAMA_CPP_SHA])
run(["git", "-C", LLAMA_DIR, "checkout", "--detach", LLAMA_CPP_SHA])
observed_sha = run(["git", "-C", LLAMA_DIR, "rev-parse", "HEAD"], capture=True).stdout.strip()
assert observed_sha == LLAMA_CPP_SHA, (observed_sha, LLAMA_CPP_SHA)

run(["cmake", "-S", LLAMA_DIR, "-B", LLAMA_DIR / "build",
     "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}",
     "-DCMAKE_BUILD_TYPE=Release"])
run(["cmake", "--build", LLAMA_DIR / "build", "--target", "llama-cli", "llama-bench", "-j2"])
LLAMA_BENCH = LLAMA_DIR / "build" / "bin" / "llama-bench"
LLAMA_CLI = LLAMA_DIR / "build" / "bin" / "llama-cli"
assert LLAMA_BENCH.is_file() and LLAMA_CLI.is_file()
print("llama.cpp pinned:", observed_sha)

In [ ]:
# No guessed KV-cache or split estimates: the measured per-GPU samples below are authoritative.
# llama.cpp reports the actual model/context allocation; begin conservatively.
contexts = [8192, 16384, 32768]
weights_gib = MODEL_PATH.stat().st_size / 1024**3
print({
    "gguf_size_gib": round(weights_gib, 2),
    "contexts_to_test": contexts,
    "note": "GGUF file size is not a VRAM estimate; inspect measured GPU peaks before increasing context."
})

In [ ]:
bench_cmd = [
    LLAMA_BENCH, "-m", MODEL_PATH, "-ngl", "99",
    "-sm", "layer", "-ts", "1,1",
    "-p", "512,2048,4096,8192", "-n", "128", "-r", "3", "-o", "json"
]
bench = run_monitored(bench_cmd, timeout=3600)
(OUTPUT_DIR / "llama-bench.json").write_text(bench.stdout or "", encoding="utf-8")
(OUTPUT_DIR / "llama-bench.stderr.txt").write_text(bench.stderr or "", encoding="utf-8")
print("llama-bench exit:", bench.returncode, "wall_seconds:", bench.wall_seconds)
print((bench.stdout or "")[-5000:])
assert bench.returncode == 0, "llama-bench failed; inspect captured stdout/stderr."

In [ ]:
context_results = []
synthetic_prompt = (
    "Write ten small Python functions with these exact names and behavior: "
    "add(a,b); clamp(x,low,high); median(values), raising ValueError on empty; "
    "slugify(text), lowercase and replace spaces with hyphens while removing punctuation; "
    "unique_preserving_order(items); is_palindrome(text), case-insensitive and ignoring spaces; "
    "parse_positive_int(value), raising ValueError unless a positive integer string; "
    "safe_divide(a,b), raising ZeroDivisionError when b=0; "
    "chunked(items,size), raising ValueError for size<=0; count_words(text), split on whitespace. "
    "Do not use imports, attributes or methods, classes, decorators, while loops, eval/exec, or dunder names. Use only direct calls to builtins and ord/chr; build strings with concatenation and character-code conversion. Return valid Python function definitions only, no markdown."
)
for ctx in contexts:
    cmd = [LLAMA_CLI, "-m", MODEL_PATH, "-ngl", "99", "-sm", "layer", "-ts", "1,1",
           "-c", str(ctx), "-n", "2048", "--temp", "0", "--seed", "42",
           "--no-display-prompt", "-p", synthetic_prompt]
    result = run_monitored(cmd, timeout=1800)
    output_path = OUTPUT_DIR / f"synthetic_ctx_{ctx}.py.txt"
    output_path.write_text(result.stdout or "", encoding="utf-8")
    (OUTPUT_DIR / f"synthetic_ctx_{ctx}.stderr.txt").write_text(result.stderr or "", encoding="utf-8")
    context_results.append({
        "context": ctx, "exit_code": result.returncode, "wall_seconds": result.wall_seconds,
        "gpu_samples": result.gpu_samples, "output_file": str(output_path)
    })
    print(context_results[-1])
    if result.returncode != 0:
        break
print(context_results)

In [ ]:
# Select the highest context profile that completed successfully.
successful = [item for item in context_results if item["exit_code"] == 0]
candidate = None
for item in reversed(successful):
    path = Path(item["output_file"])
    if path.is_file() and path.stat().st_size:
        candidate = path
        break
assert candidate is not None, "No successful synthetic generation output exists."
raw = candidate.read_text(encoding="utf-8").strip()
lines = raw.splitlines()
if lines and lines[0].lstrip().startswith("```"):
    lines = lines[1:]
if lines and lines[-1].strip() == "```":
    lines = lines[:-1]
generated = "\n".join(lines).strip()
generated_path = OUTPUT_DIR / "generated_solution.py"
generated_path.write_text(generated + "\n", encoding="utf-8")
print("Selected context output:", candidate.name)
print("Generated fixture:", generated_path)
print(generated[:6000])

In [ ]:
# Run the already-selected generated solution in a short-lived, resource-limited subprocess.
# AST allowlist rejects imports, attribute access, classes, dynamic execution and while-loops.
assert generated_path.is_file(), "Generated fixture is missing."
sandbox_runner = r'''
import ast, json, resource, sys
resource.setrlimit(resource.RLIMIT_CPU, (3, 3))
memory_limit = 512 * 1024 * 1024
resource.setrlimit(resource.RLIMIT_AS, (memory_limit, memory_limit))
source = open(sys.argv[1], "r", encoding="utf-8").read()
tree = ast.parse(source)
allowed = {
    ast.Module, ast.FunctionDef, ast.arguments, ast.arg, ast.Return, ast.Assign,
    ast.AnnAssign, ast.AugAssign, ast.If, ast.For, ast.Expr, ast.Call, ast.Name,
    ast.Load, ast.Store, ast.Constant, ast.BinOp, ast.UnaryOp, ast.Compare,
    ast.BoolOp, ast.And, ast.Or, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv,
    ast.Mod, ast.Pow, ast.UAdd, ast.USub, ast.Not, ast.Eq, ast.NotEq, ast.Lt,
    ast.LtE, ast.Gt, ast.GtE, ast.In, ast.NotIn, ast.Is, ast.IsNot, ast.List,
    ast.Tuple, ast.Set, ast.Dict, ast.Subscript, ast.Slice, ast.ListComp,
    ast.SetComp, ast.DictComp, ast.GeneratorExp, ast.comprehension, ast.Raise,
    ast.Assert, ast.Pass, ast.Break, ast.Continue, ast.IfExp, ast.keyword,
    ast.JoinedStr, ast.FormattedValue, ast.Expr
}
for node in ast.walk(tree):
    if isinstance(node, ast.Attribute):
        raise SystemExit("Attribute access is forbidden")
    if type(node) not in allowed:
        raise SystemExit("Rejected syntax node: " + type(node).__name__)
    if isinstance(node, ast.Name) and node.id.startswith("__"):
        raise SystemExit("Rejected dunder name")
function_names = {node.name for node in tree.body if isinstance(node, ast.FunctionDef)}
required_names = {
    "add", "clamp", "median", "slugify", "unique_preserving_order",
    "is_palindrome", "parse_positive_int", "safe_divide", "chunked", "count_words"
}
if not required_names.issubset(function_names):
    raise SystemExit("Missing required function(s): " + repr(sorted(required_names - function_names)))
safe_builtins = {
    "abs": abs, "all": all, "any": any, "bool": bool, "dict": dict,
    "enumerate": enumerate, "float": float, "int": int, "len": len,
    "list": list, "max": max, "min": min, "range": range, "reversed": reversed,
    "round": round, "set": set, "sorted": sorted, "str": str, "sum": sum, "ord": ord, "chr": chr,
    "tuple": tuple, "zip": zip, "ValueError": ValueError, "TypeError": TypeError,
    "ZeroDivisionError": ZeroDivisionError, "Exception": Exception
}
for node in ast.walk(tree):
    if isinstance(node, ast.Call):
        if not isinstance(node.func, ast.Name):
            raise SystemExit("Only direct function calls are allowed")
        if node.func.id not in safe_builtins and node.func.id not in function_names:
            raise SystemExit("Rejected function call: " + node.func.id)
namespace = {"__builtins__": safe_builtins}
exec(compile(tree, "generated_solution.py", "exec"), namespace, namespace)
cases = [
    ("add", lambda: namespace["add"](2, 3) == 5),
    ("clamp", lambda: namespace["clamp"](9, 0, 5) == 5),
    ("median", lambda: namespace["median"]([1, 9, 3]) == 3),
    ("slugify", lambda: namespace["slugify"]("Hola, Mundo!") == "hola-mundo"),
    ("unique_preserving_order", lambda: namespace["unique_preserving_order"]([2, 1, 2, 3]) == [2, 1, 3]),
    ("is_palindrome", lambda: namespace["is_palindrome"]("Never odd or even") is True),
    ("parse_positive_int", lambda: namespace["parse_positive_int"]("12") == 12),
    ("safe_divide", lambda: namespace["safe_divide"](9, 3) == 3),
    ("chunked", lambda: namespace["chunked"]([1, 2, 3], 2) == [[1, 2], [3]]),
    ("count_words", lambda: namespace["count_words"]("one  two\nthree") == 3),
]
results = []
for name, check in cases:
    try:
        passed = bool(check())
        detail = "PASS" if passed else "assertion returned false"
    except Exception as exc:
        passed, detail = False, type(exc).__name__ + ": " + str(exc)[:200]
    results.append({"task": name, "passed": passed, "detail": detail})
print(json.dumps(results))
'''
runner_path = OUTPUT_DIR / "sandbox_runner.py"
runner_path.write_text(sandbox_runner, encoding="utf-8")
sandbox = subprocess.run(
    [sys.executable, "-I", "-S", str(runner_path), str(generated_path)],
    capture_output=True, text=True, timeout=10, check=False
)
if sandbox.returncode == 0:
    task_results = json.loads(sandbox.stdout)
else:
    print("Sandbox rejected or failed:", sandbox.stderr[-2000:])
    task_results = [{"task": "sandbox", "passed": False, "detail": sandbox.stderr[-500:]}]
for item in task_results:
    print(item)
passed_count = sum(item["passed"] for item in task_results)
print(f"SYNTHETIC_PASS_AT_1={passed_count}/10")

In [ ]:
# Export sanitized evidence. Do not add user prompts, credentials, cookies or private files.
gpu_snapshot = run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free",
                    "--format=csv,noheader"], check=False, capture=True)
try:
    llama_bench_results = json.loads(bench.stdout)
except json.JSONDecodeError:
    llama_bench_results = {"raw_output": bench.stdout}
metrics = {
    "scope": "offline_ml_benchmark_only",
    "model": manifest,
    "model_sha256_observed": actual_sha,
    "llama_cpp_sha": observed_sha,
    "gpu_names": names,
    "compute_capability": caps,
    "gpu_memory_snapshot_after_run": gpu_snapshot.stdout,
    "llama_bench_results": llama_bench_results,
    "llama_bench": {
        "exit_code": bench.returncode,
        "wall_seconds": bench.wall_seconds,
        "gpu_samples": bench.gpu_samples,
        "stderr_tail": (bench.stderr or "")[-2000:],
    },
    "context_results": context_results,
    "synthetic_task_results": task_results,
    "synthetic_pass_at_1": passed_count,
    "policy": {"no_server": True, "no_tunnel": True, "no_core_connection": True},
}
metrics_path = OUTPUT_DIR / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print("Export:", metrics_path, "bytes:", metrics_path.stat().st_size)
print("Download metrics.json, then stop the Kaggle interactive session manually.")